In [0]:
%run "./utils/write_to_delta_utils"

In [0]:
from pyspark.sql.functions import col,coalesce,lit, lower, to_date,sum,avg,when
from delta.tables import DeltaTable

In [0]:
#---------------------------------------------------
#    Left Join sql_customer, crm_customer Dataframe
#---------------------------------------------------

df_sql_cust = spark.read.table("novamart.silver.sql_customers")
df_crm_cust = spark.read.table("novamart.silver.crm_customers")

df_dim_customers = (df_sql_cust.join(df_crm_cust, "customer_id", "left")
                   .select(
                       col("customer_id"),
                       col("first_name"),
                       col("last_name"),
                       col("full_name"),
                       col("email"),
                       col("cleaned_phone"),
                       col("address"),
                       col("city"),
                       col("region"),
                       col("customer_segment"),
                       col("verified_email"),
                       col("verified_phone"),
                       col("join_timestamp").alias("joined_at"),
                       
                       col("last_compaign_engaged"),
                       coalesce(col("lifetime_value_estimate"), lit(0.0)).alias("lifetime_value_estimate"),
                       coalesce(col("loyalty_tier"), lit("Standard")).alias("loyalty_tier"),
                       coalesce(col("preferred_channel"), lit("Email")).alias("preffered_channel"),
                       col("marketing_opt_in"),
                       col("churn_risk_score"),
                       col("crm_last_updated")
                   ))

#-------------------------------------------
#       Write to delta table dim_customers
#-------------------------------------------

write_delta_table(
    df = df_dim_customers,
    table_name = "novamart.gold.dim_customers",
    write_mode = "merge",
    merge_key = "customer_id",
    cluster_keys = ["customer_id"]
)

#-------------------------------------------
#   fact_sales dataframe
#-------------------------------------------

df_sales_silver = spark.read.table("novamart.silver.sql_sale_transactions")

df_fact_sales = (df_sales_silver
                 .filter((col("is_pricing_correct") == True) & (col("is_time_valid") == True))
                 .select(
                     col("customer_id"),
                     col("product_id"),
                     col("transaction_id"),
                     col("store_id"),
                     col("quantity"),
                     col("unit_price"),
                     col("discount_pct"),
                     col("payment_method"),
                     col("total_amount"),
                     col("transaction_timestamp"),
                     col("last_modified_timestamp")
                 ))

#-------------------------------------------
#    write to delta table fact_sales
#-------------------------------------------

write_delta_table(
    df = df_fact_sales,
    table_name = "novamart.gold.fact_sales",
    write_mode = "append",
    cluster_keys = ["cluster_id", "product_id"]
    )
#-------------------------------------------

#-------------------------------------------
#    dim_products dataframe
#-------------------------------------------
df_products_silver = spark.read.table("novamart.silver.sql_products")

df_dim_products = (df_products_silver
                   .select(
                       col("product_id"),
                       col("product_name"),
                       col("category"),
                       col("subcategory"),
                       col("brand"),
                       col("cost_price"),
                       col("unit_price"),
                       col("supplier_id"),
                       col("reorder_threshold"),
                       col("is_margin_positive"),
                       when((col("is_margin_positive") == True) & (col("unit_price") > 0),
                            ((col("unit_price") - col("cost_price")) / col("unit_price")) * 100)
                       .otherwise(0.00).cast("decimal(5,2)").alias("gross_margin_percentage"),
                       col("created_date").alias("product_created_at")
                   ))

#-------------------------------------------
#   write to delta table dim_products
#-------------------------------------------

write_delta_table(
    df = df_dim_products,
    table_name = "novamart.gold.dim_products",
    write_mode = "merge",
    merge_key = "product_id",
    cluster_keys= ["product_id"]
)
#-------------------------------------------

#-------------------------------------------
#    fact_inventory dataframe
#-------------------------------------------
df_fact_inventory = (df_products_silver
                     .select(
                         col("product_id"),
                         col("stock_quantity"),
                         col("is_reorder_needed")
                     ))

#-------------------------------------------
#    datafram clickstream
#-------------------------------------------

df_silver_clicks = spark.read.table("novamart.silver.clicks")

df_fact_clicks = (df_silver_clicks
                  .filter((col("is_event_valid") == True) & (col("is_device_valid") == True))
                  .select(
                      col("customer_id"),
                      col("product_id"),
                      col("session_id"),
                      col("event_id"),
                      col("event_type"),
                      col("device_type"),
                      col("page_url"),
                      col("search_query"),
                      col("event_timestamp").alias("event_at")
                     ))

#-------------------------------------------
#    datafram clickstream
#-------------------------------------------
write_delta_table(
    df = df_fact_clicks,
    table_name = "novamart.gold.fact_clicks",
    write_mode = "append",
    cluster_keys = ["customer_id"]
)

In [0]:
df_dim_cust = spark.read.table("novamart.gold.dim_customers")
df_dim_product = spark.read.table("novamart.gold.dim_products")
df_fact_sales = spark.read.table("novamart.gold.fact_sales")

df_joined = (
    df_fact_sales
    .join(df_dim_cust, on = "customer_id", how = "left")
    .join(df_dim_product, on = "product_id", how = "left")
)

df_agg_daily_sales = (df_joined
                      .groupBy(
                          to_date(col("transaction_timestamp")).alias("sales_date"),
                          col("category").alias("product_category"),
                          col("region").alias("customer_region")
                          )
                      .agg(
                          sum(col("total_amount")).cast("decimal(12,2)").alias("daily_sale"),
                          sum(col("quantity").cast("integer")).alias("daily_units_sold"),
                          avg(col("gross_margin_percentage").cast("decimal(12,2)")).alias("average_margin_pct")
                          )
                      )

write_delta_table(
    df = df_agg_daily_sales,
    table_name = "novamart.gold.fact_daily_sales",
    write_mode = "merge",
    merge_key = ["sales_date", "product_category", "customer_region"],
    cluster_keys = ["sales_date", "product_category", "customer_region"]
)

In [0]:
df_fact_inventory = spark.read.table("novamart.gold.fact_inventory")
df_fact_clicks = spark.read.table("novamart.gold.fact_clicks")
df_dim_products = spark.read.table("novamart.gold.dim_products")

df_add_to_cart = (df_fact_clicks.filter(col("event_type") == "ADD_TO_CART"))

df_low_stock_alert = (df_add_to_cart
                      .join(df_fact_inventory, on = "product_id", how="inner")
                      .join(df_dim_products, on="product_id", how="inner")
                      .filter(col("stock_quantity") <= col("reorder_threshold"))
                      .select(
                          col("event_id"),
                          col("session_id"),
                          col("product_id"),
                          col("stock_quantity").alias("available_stock"),
                          col("event_at").alias("alert_timestamp")
                      ))

write_delta_table(
    df= df_low_stock_alert,
    table_name="novamart.gold.fact_low_stock_alert",
    write_mode="append",
    cluster_keys=["product_id", "alert_timestamp"]
)